# 01. Data preparation

This notebook creates the reproducible cohort, feature sets and train/test split used by both TabPFN-3.5 models.

**Design choice:** the final app uses a curated core feature set: the variables marked `ForceInc=1` in the supplied dictionary plus eight demographic/lifestyle variables. This keeps the model clinically interpretable and the interactive app manageable.

Age leakage is detected from the human-readable dictionary rather than relying on a hand-written list alone.

In [3]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

# Locate the repository root even if Jupyter was launched inside notebooks/
root = Path.cwd().resolve()
for candidate in [root, *root.parents]:
    if (candidate / "src").exists() and (candidate / "data").exists():
        root = candidate
        break
sys.path.insert(0, str(root))

from src.config import (
    NHANES_DATA_PATH, VARIABLE_DICT_PATH, CATEGORICAL_VARIABLES_PATH, PROCESSED_DIR, DEMO_DIR, RESULTS_DIR,
    RANDOM_STATE, TEST_SIZE, ADULT_MIN_AGE, TARGET_AGE, TARGET_MORTALITY,
    DESIGN_COLS, MORTALITY_COLS
)
from src.data_utils import load_nhanes, load_variable_dictionary, numeric_coercion, build_feature_sets, load_categorical_variable_names, save_json

print("Project root:")
print("Python:", sys.version.split()[0])

Project root:
Python: 3.11.17


In [4]:
# Environment sanity check — useful because the hosted client serializes tables through PyArrow.
import pyarrow
import tabpfn_client

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("tabpfn-client:", tabpfn_client.__version__)

# The repository pins pandas 2.3.3 / pyarrow 23.0.1 / tabpfn-client 0.6.1.
# If you see ArrowKeyError about pandas.period, restart the Jupyter kernel after installing requirements.txt.

Python: 3.11.17 | packaged by Anaconda, Inc. | (main, Oct  1 2026, 20:16:54) [MSC v.1942 64 bit (AMD64)]
NumPy: 2.4.6
Pandas: 2.3.3
tabpfn-client: 0.6.1


In [5]:
variables = load_variable_dictionary(VARIABLE_DICT_PATH)
nhanes = load_nhanes(NHANES_DATA_PATH)
nhanes = numeric_coercion(nhanes)

print("NHANES shape:", nhanes.shape)
print("Dictionary shape:", variables.shape)
print(variables["Demo/Exam/Quest/Lab/Mort"].value_counts())

missing_dictionary_vars = sorted(set(variables["Var"]) - set(nhanes.columns))
missing_nhanes_vars = sorted(set(nhanes.columns) - set(variables["Var"]))
print("Dictionary variables not present in NHANES:", len(missing_dictionary_vars))
print(missing_dictionary_vars[:20])
print("NHANES variables not present in Dictionary:", len(missing_nhanes_vars))
print(missing_nhanes_vars[:20])

NHANES shape: (21004, 1171)
Dictionary shape: (1144, 9)
Demo/Exam/Quest/Lab/Mort
Q            609
E            267
LAB          224
DEMO          37
MORTALITY      7
Name: count, dtype: int64
Dictionary variables not present in NHANES: 170
['BPQ042', 'BPQ045', 'BPQ055', 'BPQ095', 'BPQ115', 'BPQ117', 'BPQ125', 'BPQ135', 'DIQ065', 'DIQ075', 'DIQ085', 'HUD070', 'HUQ080', 'HUQ081', 'HUQ088B', 'HUQ088C', 'HUQ089', 'KID182', 'KIDUQUEX', 'KIQ022']
NHANES variables not present in Dictionary: 209
['DED011', 'DED021', 'DED031', 'DED051', 'DED080A', 'DED080B', 'DED080C', 'DED080D', 'DED080E', 'DED080F', 'DED080G', 'DED080H', 'DED080I', 'DED080J', 'DED082A', 'DED082B', 'DED082C', 'DED082D', 'DED082E', 'DED082F']


In [6]:
# Build the feature sets from the supplied dictionary.
feature_sets = build_feature_sets(variables, nhanes.columns)

print("Age leakage columns:", len(feature_sets["age_leakage"]))
print("Core age features:", len(feature_sets["age_core"]))
print("Wide age features available:", len(feature_sets["age_wide"]))
print("Mortality core features:", len(feature_sets["mortality_core"]))
print("\nFirst core age features:", feature_sets["age_core"][:30])

Age leakage columns: 75
Core age features: 71
Wide age features available: 881
Mortality core features: 72

First core age features: ['BPXPLS', 'BPXSAR', 'BPXDAR', 'BMXBMI', 'BMXWAIST', 'BMXTRI', 'URXUMASI', 'URXUCRSI', 'LBDIRNSI', 'LBDTIBSI', 'LBXPCT', 'LBDFERSI', 'LBDFOLSI', 'LBDB12SI', 'LBXHCY', 'LBXMMA', 'LBDRBFSI', 'LBDTCSI', 'LBDHDLSI', 'LBXWBCSI', 'LBXLYPCT', 'LBXMOPCT', 'LBXNEPCT', 'LBXEOPCT', 'LBXBAPCT', 'LBDLYMNO', 'LBDMONO', 'LBDNENO', 'LBDEONO', 'LBDBANO']


In [7]:
# Define the analytic adult cohort.
# RIDAGEYR is top-coded at 85 in NHANES; we keep adults >=20 and document the top-code caveat.
cohort = nhanes.loc[nhanes[TARGET_AGE].between(ADULT_MIN_AGE, 120, inclusive="both")].copy()
cohort[TARGET_AGE] = pd.to_numeric(cohort[TARGET_AGE], errors="coerce")
cohort[TARGET_MORTALITY] = pd.to_numeric(cohort[TARGET_MORTALITY], errors="coerce")
cohort = cohort.loc[cohort[TARGET_AGE].notna() & cohort[TARGET_MORTALITY].isin([0, 1])].copy()

# Stable row identifier for all downstream joins. This is not used as a model feature.
cohort = cohort.reset_index(drop=True)
cohort["ROW_ID"] = np.arange(len(cohort), dtype=int)

print("Adult cohort:", cohort.shape)
print("Mortality prevalence:", cohort[TARGET_MORTALITY].mean())
print("Age range:", cohort[TARGET_AGE].min(), cohort[TARGET_AGE].max())

Adult cohort: (10282, 1172)
Mortality prevalence: 0.3181287687220385
Age range: 20 85


In [8]:
# Reproducible shared train/test split.
# Stratifying on mortality keeps the downstream classification problem balanced across the split.
from sklearn.model_selection import train_test_split

train_ids, test_ids = train_test_split(
    cohort["ROW_ID"].to_numpy(),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=cohort[TARGET_MORTALITY]
)

train_ids = np.sort(train_ids)
test_ids = np.sort(test_ids)

print("Train rows:", len(train_ids))
print("Test rows:", len(test_ids))

Train rows: 8225
Test rows: 2057


In [11]:
# Keep only model columns plus ROW_ID for traceability.
age_features = feature_sets["age_core"]
mortality_features = feature_sets["mortality_core"]

# Categorical variables from the supplied NHANES categorical dictionary.
categorical_variables = load_categorical_variable_names(CATEGORICAL_VARIABLES_PATH)
categorical_age_features = [c for c in age_features if c in categorical_variables]
categorical_mortality_features = [c for c in mortality_features if c in categorical_variables]

print("Categorical age features:", categorical_age_features)
print("Categorical mortality features:", categorical_mortality_features)

train_df = cohort.loc[cohort.ROW_ID.isin(train_ids)].copy()
test_df = cohort.loc[cohort.ROW_ID.isin(test_ids)].copy()

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DEMO_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Save model matrices. Targets are kept separate from X in the next notebooks.
train_df[["ROW_ID", TARGET_AGE, TARGET_MORTALITY] + mortality_features].to_csv(
    PROCESSED_DIR / "train_mortality.csv", index=False
)
test_df[["ROW_ID", TARGET_AGE, TARGET_MORTALITY] + mortality_features].to_csv(
    PROCESSED_DIR / "test_mortality.csv", index=False
)
train_df[["ROW_ID", TARGET_AGE, TARGET_MORTALITY] + age_features].to_csv(
    PROCESSED_DIR / "train_age.csv", index=False
)
test_df[["ROW_ID", TARGET_AGE, TARGET_MORTALITY] + age_features].to_csv(
    PROCESSED_DIR / "test_age.csv", index=False
)

metadata = {
    "age_features": age_features,
    "mortality_features": mortality_features,
    "age_leakage": feature_sets["age_leakage"],
    "mortality_columns": feature_sets["mortality_columns"],
    "train_rows": train_ids.tolist(),
    "test_rows": test_ids.tolist(),
    "target_age": TARGET_AGE,
    "target_mortality": TARGET_MORTALITY,
    "random_state": RANDOM_STATE,
    "categorical_age_features": categorical_age_features,
    "categorical_mortality_features": categorical_mortality_features,
}
save_json(metadata, PROCESSED_DIR / "metadata.json")

print("Saved processed data and metadata.")

Categorical age features: ['RIAGENDR', 'RIDRETH1', 'DMDEDUC2', 'DMDMARTL', 'SMQ040', 'PAQ180']
Categorical mortality features: ['RIAGENDR', 'RIDRETH1', 'DMDEDUC2', 'DMDMARTL', 'SMQ040', 'PAQ180']
Saved processed data and metadata.


In [12]:
# Create small demo-training tables.
# They let the Streamlit app fall back to a fresh TabPFN-3.5 fit if the saved
# server-side model record is unavailable to the current account.
DEMO_N = min(800, len(train_df))
rng = np.random.default_rng(RANDOM_STATE)
demo_ids = rng.choice(train_df.index.to_numpy(), size=DEMO_N, replace=False)

demo_age = train_df.loc[demo_ids, age_features + [TARGET_AGE]].copy()
demo_mort = train_df.loc[demo_ids, mortality_features + [TARGET_MORTALITY]].copy()

demo_age.to_csv(DEMO_DIR / "train_age_demo.csv", index=False)
demo_mort.to_csv(DEMO_DIR / "train_mortality_demo.csv", index=False)
print("Demo training rows:", DEMO_N)

Demo training rows: 800


In [13]:
# Summaries used by the app for sensible numeric bounds/defaults.
summary = {}
for col in mortality_features:
    s = pd.to_numeric(train_df[col], errors="coerce")
    valid = s.dropna()
    if len(valid) == 0:
        continue
    summary[col] = {
        "median": float(valid.median()),
        "min": float(valid.min()),
        "max": float(valid.max()),
    }

save_json(summary, PROCESSED_DIR / "feature_summary.json")
print("Saved feature_summary.json")

Saved feature_summary.json
